# 00 - Create SWSAcademy Advanced ML Jobs

Run this notebook **once** after importing the SWSAcademy course folder. It creates one Lakeflow job per Advanced
Machine Learning folder that runs that folder's **demos** in order (labs and the Capstone are not included):

| Job | Compute | What it runs |
|---|---|---|
| SWSAcademy - A0 Advanced ML Concepts (Serverless) | Serverless | The nine concept notebooks: Pandas on Spark, Spark ML, distributed patterns, testing, CI/CD, rollout, drift, bundles |
| SWSAcademy - A1 Model Development at Scale | Your cluster | Admin setup, Pandas API on Spark, Spark ML + MLflow, Spark and Ray resources, Ray Train |
| SWSAcademy - A2 Advanced ML Operations | Your cluster | Admin setup, testing strategies, integration-test pipeline, rollout with Model Serving, Lakehouse Monitoring, Asset Bundles, and CI/CD with Git (when GitHub credentials are saved) |

**Before you run it**
- Attach this notebook to the **classic cluster** the A1 and A2 jobs should use: a single-node cluster with
  **Databricks Runtime 16.4 LTS ML** (for example 4 cores, 16 GB). The jobs reuse that cluster; they do not create one.
- For the CI/CD demo (A2 M01 · 1.1), first run **M01 · 0 - Generate Tokens** in the A2 folder to save your GitHub
  username, repository and token. Without it, the A2 job is created without the CI/CD task.

The notebook finds the course notebooks **relative to its own location**, so it works wherever the folder was imported.
Running it again updates the jobs instead of creating duplicates.

In [ ]:
dbutils.widgets.text("name_suffix", "", "Job name suffix (optional, e.g. your initials)")
dbutils.widgets.text("cluster_id", "", "Cluster ID for A1/A2 (empty = the cluster this notebook is attached to)")

In [ ]:
import hashlib, os, re
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
ROOT = here.rsplit("/", 1)[0]
ADV = f"{ROOT}/02 Advanced Machine Learning"
SUFFIX = dbutils.widgets.get("name_suffix").strip()
TAGS = {"course": "swsacademy", "purpose": "instructor-demo"}

A0 = f"{ADV}/0 - Advanced ML Concepts (Serverless)"
A1 = f"{ADV}/1 - swsacademy-model-development-at-scale"
A2 = f"{ADV}/2 - swsacademy-advanced-machine-learning-operations"

def exists(path):
    try:
        w.workspace.get_status(path)
        return True
    except Exception:
        return False

missing = [p for p in (A0, A1, A2) if not exists(p)]
if missing:
    raise FileNotFoundError("These course folders were not found next to this notebook:\n  " + "\n  ".join(missing)
                            + "\nKeep this notebook in the top folder that contains '02 Advanced Machine Learning'.")
print("Course folder:", ROOT)

## 1. Choose the cluster for A1 and A2

In [ ]:
cluster_id = dbutils.widgets.get("cluster_id").strip()
if not cluster_id:
    if os.environ.get("IS_SERVERLESS", "").lower() == "true":
        raise ValueError("This notebook is running on serverless. Attach it to your 16.4 LTS ML cluster, "
                         "or paste that cluster's ID into the 'Cluster ID' box at the top, then run it again.")
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")

cluster = w.api_client.do("GET", "/api/2.1/clusters/get", query={"cluster_id": cluster_id})   # REST: works with any SDK version
runtime = cluster.get("spark_version", "")
is_ml = bool(cluster.get("use_ml_runtime")) or "-ml-" in runtime
print(f"Cluster: {cluster.get('cluster_name')} ({cluster_id}), runtime {runtime}{' ML' if is_ml else ''}")
if not is_ml or not runtime.startswith("16.4"):
    print("WARNING: the Advanced ML demos were tested on Databricks Runtime 16.4 LTS ML. "
          "Other runtimes may fail; change the cluster's runtime or pass another cluster ID.")

## 2. Create or update the jobs

In [ ]:
def nb(key, path, deps=(), params=None, classic=True):
    task = {"task_key": key, "notebook_task": {"notebook_path": path, "source": "WORKSPACE"}, "timeout_seconds": 3600}
    if params:
        task["notebook_task"]["base_parameters"] = params
    if classic:
        task["existing_cluster_id"] = cluster_id
    if deps:
        task["depends_on"] = [{"task_key": d} for d in deps]
    return task

def chain(items, classic=True):
    """items: (key, path) or (key, path, params); each task depends on the previous one."""
    tasks, prev = [], None
    for key, path, *params in items:
        tasks.append(nb(key, path, [prev] if prev else (), params[0] if params else None, classic))
        prev = key
    return tasks

# The A2 integration-test pipeline (M02 · 2.2.1) takes the learner schema that the A2 admin setup creates
owner = spark.sql("SELECT current_user() AS u").first()["u"]
slug = re.sub(r"[^a-zA-Z0-9_]", "_", owner.split("@")[0]).lower().strip("_") or "learner"
SCHEMA = f"learner_{slug[:40]}_{hashlib.sha1(owner.encode()).hexdigest()[:8]}"
m01, m02, p221 = (f"{A2}/M01 - Streamlining MLOps with Databricks", f"{A2}/M02 - Model Rollout Strategies with Databricks",
                  f"{A2}/M02 - Model Rollout Strategies with Databricks/2.2.1 - Model Training Pipeline")
pipe = {"catalog": "swsacademy", "schema": SCHEMA}

a2_items = [
    ("00_admin_setup", f"{A2}/00 - SWSAcademy Admin Setup"),
    ("2_1_testing_strategies", f"{m02}/2.1 Demo - Common Testing Strategies"),
    ("2_2_1_silver_to_feature_store", f"{p221}/01 Silver to Feature Store",
     dict(pipe, silver_table_name="diabetes", column="Age", target_column="Diabetes_binary", primary_key="id")),
    ("2_2_2_features_validation", f"{p221}/02 Features Validation", dict(pipe, normalized_column="Age")),
    ("2_2_3_train_validated", f"{p221}/03  Train Model on Validated Features",
     dict(pipe, primary_key="id", target_column="Diabetes_binary", username=owner, silver_table_name="diabetes")),
    ("2_3_rollout_model_serving", f"{m02}/2.3 Demo - Model Rollout Strategies with Mosaic AI Model Serving"),
    ("3_1_lakehouse_monitoring", f"{A2}/M03 - Lakehouse Monitoring/3.1 Demo - Monitoring Model Quality"),
    ("4_1_asset_bundles", f"{A2}/M04 - Build ML Assets as Code/4.1 Demo - Working with Asset Bundles"),
]
has_git = exists(f"{m01}/var/git_credentials.cfg")
if has_git:
    a2_items.append(("1_1_cicd_with_git", f"{m01}/1.1 Demo - Building a CI-CD Pipeline with Databricks CLI"))

d03 = f"{A1}/03 - Distributed Model Training on Databricks"
JOBS = [
    ("SWSAcademy - A0 Advanced ML Concepts (Serverless)",
     "Advanced ML concept notebooks on serverless: Pandas on Spark, Spark ML, distributed patterns, testing, CI/CD, rollout, drift, bundles.",
     chain([
        ("00_course_map", f"{A0}/00 - Course Map"),
        ("01_pandas_on_spark", f"{A0}/01 - Pandas API on Spark"),
        ("02_spark_ml_mlflow", f"{A0}/02 - Spark ML at Scale with MLflow"),
        ("03_distributed_patterns", f"{A0}/03 - Distributed Training Patterns"),
        ("04_testing", f"{A0}/04 - Testing ML Pipelines"),
        ("05_cicd_jobs_as_code", f"{A0}/05 - CI-CD with Jobs as Code"),
        ("06_rollout", f"{A0}/06 - Rollout Strategies"),
        ("07_monitoring_drift", f"{A0}/07 - Monitoring and Drift"),
        ("08_asset_bundles", f"{A0}/08 - Asset Bundles"),
     ], classic=False)),
    ("SWSAcademy - A1 Model Development at Scale",
     "Folder A1 demos: Pandas API on Spark, Spark ML + MLflow, Spark and Ray resources, Ray Train.",
     chain([
        ("00_admin_setup", f"{A1}/00 - SWSAcademy Admin Setup"),
        ("01_1_pandas_on_spark", f"{A1}/01 - Foundations of Model Development/01.1 Demo - Scalable Data Transformation Using Pandas API on Spark"),
        ("02_1_sparkml_mlflow", f"{A1}/02 - Scalable Model Development on Databricks/02.1 Demo - Scalable Model Training with SparkML and MLflow Tracking"),
        ("03_1_spark_ray_resources", f"{d03}/03.1 Demo - Understanding Spark and Ray Resources on Databricks"),
        ("03_2_ray_train", f"{d03}/03.2 Demo - Scalable Model Training with Ray Train"),
     ])),
    ("SWSAcademy - A2 Advanced ML Operations",
     "Folder A2 demos: testing, integration-test pipeline, rollout with Model Serving, Lakehouse Monitoring, Asset Bundles"
     + (", CI/CD with Git." if has_git else ". (CI/CD with Git is added after GitHub credentials are saved in M01 · 0 - Generate Tokens.)"),
     chain(a2_items)),
]

# check every notebook a job points at before creating anything
bad = [t["notebook_task"]["notebook_path"] for _, _, tasks in JOBS for t in tasks if not exists(t["notebook_task"]["notebook_path"])]
if bad:
    raise FileNotFoundError("Notebooks not found:\n  " + "\n  ".join(bad))

existing = {j.settings.name: j.job_id for j in w.jobs.list() if j.settings and j.settings.name}
host = w.config.host.rstrip("/")
results = []

def upsert(name, settings):
    if name in existing:
        w.api_client.do("POST", "/api/2.1/jobs/reset", body={"job_id": existing[name], "new_settings": settings})
        return existing[name], "updated"
    return w.api_client.do("POST", "/api/2.1/jobs/create", body=settings)["job_id"], "created"

for name, desc, tasks in JOBS:
    full = f"{name}{' ' + SUFFIX if SUFFIX else ''}"
    settings = {"name": full, "description": desc, "tags": TAGS, "max_concurrent_runs": 1,
                "queue": {"enabled": True}, "tasks": tasks}
    results.append((full, len(tasks)) + upsert(full, settings))

html = "<table><tr><th style='text-align:left'>Job</th><th>Tasks</th><th>Status</th></tr>" + "".join(
    f"<tr><td><a href='{host}/jobs/{jid}' target='_blank'>{n}</a></td><td style='text-align:center'>{k}</td><td>{s}</td></tr>"
    for n, k, jid, s in results) + "</table>"
note = "" if has_git else ("<p><b>Note:</b> the A2 job has no CI/CD task yet. Run <i>M01 · 0 - Generate Tokens</i> in the A2 folder, "
                           "then run this notebook again to add it.</p>")
displayHTML(html + note + "<p>Run the jobs <b>one at a time</b>: they share one cluster. "
            "A0 runs on serverless and needs the <code>swsacademy</code> catalog and Diabetes data from the A2 admin setup, "
            "so run A2 (or its admin setup) once before A0. "
            "<b>Cost:</b> A2 leaves a Model Serving endpoint running (2.3); delete it after class.</p>")